# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [5]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [6]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [7]:
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender

optimizer = ModelOptimizer("EASE_R")

STUDY_NAME = EASE_R_Recommender.RECOMMENDER_NAME

In [8]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "topK": optuna_trial.suggest_int("topK", 10, 1500),
        "l2_norm": optuna_trial.suggest_float("l2_norm", 1e0, 1e5, log=True),
        "normalize_matrix": optuna_trial.suggest_categorical("normalize_matrix", [True, False]),
    }
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = EASE_R_Recommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [9]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=25
)

[I 2025-11-20 20:56:25,762] Using an existing study with name 'EASE_R_Recommender' instead of creating a new one.


  0%|          | 0/25 [00:00<?, ?it/s]

EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 6.87 sec
  Fold 1/5 - Score: 0.22442509233951569
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 6.95 sec
  Fold 2/5 - Score: 0.2257239669561386
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 7.23 sec
  Fold 3/5 - Score: 0.22664394974708557
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.44 sec
  Fold 4/5 - Score: 0.22515279054641724
[I 2025-11-20 20:57:37,155] Trial 80 finished with value: 0.22548644244670868 and parameters: {'topK': 1242, 'l2_norm': 348.4401793472633, 'normalize_matrix': True}. Best is trial 25 with value: 0.28151339292526245.
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 7.92 sec
  Fold 1/5 - Score: 0.26892974972724915
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 7.73 sec
  Fold 2/5 - Score: 0.26712748408

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- ADD HERE